# PCA highlighted with covariates

In [ ]:
import os
# Project data root. Override with the ACC_DATA_ROOT environment variable.
ACC_DATA_ROOT = os.environ.get("ACC_DATA_ROOT", os.path.expanduser("~/acc_data"))


In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# import required packages
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

In [ ]:
adata = sc.read_h5ad(f'{ACC_DATA_ROOT}/bulk_rna_data/h5ad_files/bulk_counts.h5ad')

In [ ]:
adata

In [ ]:
adata.var['Symbol']

In [ ]:
# I/O configuration
IN  = f"{ACC_DATA_ROOT}/bulk_rna_data/h5ad_files/bulk_counts.h5ad"
OUT = Path(f"{ACC_DATA_ROOT}/bulk_rna_data/outputs/images/clinical_outcome"); 
# OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
# standard plotting setting summarized.
sc.settings.verbosity = 2
sc.settings.figdir = str(OUT)
sc.settings.set_figure_params(dpi=120, format="svg", facecolor="white")
plt.rcParams.update({"font.family": "sans-serif", "font.size": 11, "axes.linewidth": 0.8})

In [ ]:
# load and see adata
adata = sc.read_h5ad(IN)
print(adata)

In [ ]:
adata.obs

In [ ]:
adata.obs['ClinicalOutcome'] = adata.obs['ClinicalOutcome'].map({'5Plus':'Good', '2Minus':'Poor'})
adata.obs['ModelOutcome2'] = adata.obs['ModelOutcome2'].map({'Average':'Good', 'Poor':'Poor'})
adata.obs['ModelOutcome1'] = adata.obs['ModelOutcome1'].map({'Average':'Good', 'Poor':'Poor'})

In [ ]:
# Merge BOT and Baseoftongue into a single "Base of Tongue" category
adata.obs['Site'] = adata.obs['Site'].replace({'BOT': 'Base of Tongue', 'Baseoftongue': 'Base of Tongue'})

In [ ]:
adata.obs

In [ ]:
# Use gene symbols if available; fallback to gene_id
if "Symbol" in adata.var.columns:
    sym = adata.var["Symbol"].astype(str)
    if "gene_id" in adata.var.columns:
        sym = sym.where(sym.ne("nan"), adata.var["gene_id"].astype(str))
    adata.var_names = sym.values
    adata.var_names_make_unique()
elif "gene_id" in adata.var.columns:
    adata.var_names = adata.var["gene_id"].astype(str).values
    adata.var_names_make_unique()

In [ ]:
# changing all columns as categorical so its easier to plot.
for c in ["Site","ClinicalOutcome","DEGOutcome","ModelOutcome1","ModelOutcome2"]:
    if c in adata.obs: adata.obs[c] = adata.obs[c].astype("category")
if "ClinicalOutcome" in adata.obs and set(adata.obs["ClinicalOutcome"].cat.categories) >= {"Good","Poor"}:
    adata.obs["ClinicalOutcome"] = adata.obs["ClinicalOutcome"].cat.reorder_categories(["Good","Poor"], ordered=True)


In [ ]:
# Keep raw counts
adata.layers["counts"] = adata.X.copy()

In [ ]:
# I dont want to subset genes, conisder everything.

# # -------- Gene filtering (bulk-appropriate) --------
# # Keep genes with CPM > 1 in ≥ 20% of samples
# libsize = np.asarray(adata.layers["counts"].sum(axis=1)).ravel()
# counts = adata.layers["counts"]
# cpm = (counts / libsize[:, None]) * 1e6
# try: cpm = np.array(cpm)
# except: cpm = cpm.A
# keep = (cpm > 1.0).sum(axis=0) >= max(1, int(0.2 * adata.n_obs))
# adata = adata[:, keep].copy()
# print(f"Kept {adata.n_vars} genes after CPM>1 in ≥20% samples.")

In [ ]:
# -------- Normalization (pure Scanpy) --------
sc.pp.normalize_total(adata, target_sum=1e6)  # CPM
sc.pp.log1p(adata)                            # log1p CPM
adata.raw = adata

# -------- HVGs, scale, PCA --------
sc.pp.highly_variable_genes(adata, n_top_genes=3000, flavor="seurat_v3", subset=True)
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, n_comps=10, svd_solver="arpack")

# Export PCA matrices
pd.DataFrame(
    adata.obsm["X_pca"], index=adata.obs_names,
    columns=[f"PC{i+1}" for i in range(adata.obsm['X_pca'].shape[1])]
).to_csv(OUT/"pca_scores_samples.csv")

pd.DataFrame(
    adata.varm["PCs"], index=adata.var_names,
    columns=[f"PC{i+1}" for i in range(adata.varm['PCs'].shape[1])]
).to_csv(OUT/"pca_loadings_genes.csv")

# Plot helpers
def savefig_dual(name):
    plt.savefig(OUT/f"{name}.svg", bbox_inches="tight")

def strip_frame(fig):
    """Keep only the left/bottom (X/Y) axes; drop the top and right spines."""
    for ax in fig.axes:
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_visible(True)
        ax.spines["bottom"].set_visible(True)

# PCA colored by covariates
var_ratio = adata.uns["pca"]["variance_ratio"]  # fraction of total variance per PC
# for c in [x for x in ["Site","ClinicalOutcome","DEGOutcome","ModelOutcome1","ModelOutcome2"] if x in adata.obs]:
for c in [x for x in ["Site","ClinicalOutcome", "ModelOutcome1", "ModelOutcome2"] if x in adata.obs]:
    fig = sc.pl.pca(
        adata, color=c, size=130, legend_loc="right margin",
        alpha=0.9, show=False, return_fig=True,
        title=f"PCA by {c}  [log1p(CPM); {adata.n_vars} HVGs]"
    )
    ax = fig.axes[0]
    strip_frame(fig)
    ax.set_xlabel(f"PC1 ({var_ratio[0]*100:.1f}% variance)")
    ax.set_ylabel(f"PC2 ({var_ratio[1]*100:.1f}% variance)")
    savefig_dual(f"pca_by_{c}")

sc.pl.pca_variance_ratio(adata, n_pcs=10, show=False)
strip_frame(plt.gcf())
savefig_dual("pca_variance_ratio")

In [ ]:
# -------- ClinicalOutcome PCA: variance-% axis labels --------
color_key = "ClinicalOutcome"
var_ratio = adata.uns["pca"]["variance_ratio"]  # fraction of total variance per PC

fig = sc.pl.pca(
    adata, color=color_key, size=130, legend_loc="right margin",
    alpha=0.9, show=False, return_fig=True,
    title=f"PCA by {color_key}  [log1p(CPM); {adata.n_vars} HVGs]"
)
ax = fig.axes[0]
strip_frame(fig)

ax.set_xlabel(f"PC1 ({var_ratio[0]*100:.1f}% variance)")
ax.set_ylabel(f"PC2 ({var_ratio[1]*100:.1f}% variance)")

savefig_dual(f"pca_by_{color_key}_annotated")

In [ ]:
# -------- ClinicalOutcome PCA: higher-order PC pairs (PC3/PC4, PC5/PC6, ...) --------
def plot_pca_pair(pcx, pcy, color_key="ClinicalOutcome"):
    """Same styling as the PC1/PC2 plot above (variance-% labels),
    generalized to an arbitrary pair of PCs via scanpy's `components` argument."""
    var_ratio = adata.uns["pca"]["variance_ratio"]

    fig = sc.pl.pca(
        adata, color=color_key, components=f"{pcx},{pcy}", size=130, legend_loc="right margin",
        alpha=0.9, show=False, return_fig=True,
        title=f"PCA by {color_key}  [log1p(CPM); {adata.n_vars} HVGs]"
    )
    ax = fig.axes[0]
    strip_frame(fig)

    ax.set_xlabel(f"PC{pcx} ({var_ratio[pcx-1]*100:.1f}% variance)")
    ax.set_ylabel(f"PC{pcy} ({var_ratio[pcy-1]*100:.1f}% variance)")

    savefig_dual(f"pca_by_{color_key}_PC{pcx}_PC{pcy}")
    return fig

# PC1/PC2 is already plotted in the cell above; sweep the remaining consecutive
# pairs out to however many components were computed (n_comps=10 above).
n_comps = adata.obsm["X_pca"].shape[1]
for pcx, pcy in zip(range(3, n_comps, 2), range(4, n_comps + 1, 2)):
    plot_pca_pair(pcx, pcy)

In [ ]:
# can I do all of this in a simpler way without over the top error correction code

# 1. Adding the Gene ID as column names.
adata.var_names = adata.var['Symbol']

In [ ]:
adata.var_names

In [ ]:
# # changing all columns as categorical so its easier to plot.
# for c in ["Site","ClinicalOutcome","DEGOutcome","ModelOutcome1","ModelOutcome2"]:
#     if c in adata.obs: adata.obs[c] = adata.obs[c].astype("category")
# if "ClinicalOutcome" in adata.obs and set(adata.obs["ClinicalOutcome"].cat.categories) >= {"Good","Poor"}:
#     adata.obs["ClinicalOutcome"] = adata.obs["ClinicalOutcome"].cat.reorder_categories(["Good","Poor"], ordered=True)


# Essential Code which does the same. Can follow this.

In [ ]:
# # Use gene symbols if available; fallback to gene_id
# if "Symbol" in adata.var.columns:
#     sym = adata.var["Symbol"].astype(str)
#     if "gene_id" in adata.var.columns:
#         sym = sym.where(sym.ne("nan"), adata.var["gene_id"].astype(str))
#     adata.var_names = sym.values
#     adata.var_names_make_unique()
# elif "gene_id" in adata.var.columns:
#     adata.var_names = adata.var["gene_id"].astype(str).values
#     adata.var_names_make_unique()


# adata.var_names


# adata.obs['ClinicalOutcome'] = adata.obs['ClinicalOutcome'].map({'5Plus':'Good', '2Minus':'Poor'})
# adata.obs['DEGOutcome'] = adata.obs['DEGOutcome'].map({'5Plus':'Good', '2Minus':'Poor'})
# adata.obs['ModelOutcome1'] = adata.obs['ModelOutcome1'].map({'Average':'Good', 'Poor':'Poor'})
# adata.obs['ModelOutcome2'] = adata.obs['ModelOutcome2'].map({'Average':'Good', 'Poor':'Poor'})


# set(list(adata.obs['Site']))


# # correct site mapper
# sites = {
#     'BOT':'Base of Tongue',
#     'Baseoftongue':'Base of Tongue',
#     'BuccalMucosa':'Buccal Mucosa',
#     'FloorofMouth':'Floor of Mouth',
#     'Larynx':'Larynx',
#     'Mandible':'Mandible',
#     'Maxilla':'Maxilla',
#     'Maxillarysinus':'Maxillary Sinus',
#     'Parotid':'Parotid Gland',
#     'Submandibular':'Submandibular Gland',
#     'Tongue':'Tongue'
# }


# adata.obs['Site'] = adata.obs['Site'].map(sites)


# # changing all columns as categorical so its easier to plot and ordering it well for plotting
# for c in ["Site","ClinicalOutcome","DEGOutcome","ModelOutcome1","ModelOutcome2"]:
#     if c in adata.obs: adata.obs[c] = adata.obs[c].astype("category")
# if "ClinicalOutcome" in adata.obs and set(adata.obs["ClinicalOutcome"].cat.categories) >= {"Good","Poor"}:
#     adata.obs["ClinicalOutcome"] = adata.obs["ClinicalOutcome"].cat.reorder_categories(["Good","Poor"], ordered=True)


# adata.layers["counts"] = adata.X.copy()


# # -------- Normalization (pure Scanpy) --------
# sc.pp.normalize_total(adata, target_sum=1e6)  # CPM
# sc.pp.log1p(adata)                            # log1p CPM
# adata.raw = adata

# # -------- HVGs, scale, PCA --------
# sc.pp.highly_variable_genes(adata, n_top_genes=3000, flavor="seurat_v3", subset=True)
# sc.pp.scale(adata, max_value=10)
# sc.tl.pca(adata, n_comps=10, svd_solver="arpack")


# cov = ["Site","ClinicalOutcome","DEGOutcome","ModelOutcome1","ModelOutcome2"]

# fig, ax = plt.subplots(figsize = (6, 6))
# sc.pl.pca(
#     adata,
#     color = ['ModelOutcome2'],
#     size = 200,
#     legend_loc = 'right margin',
#     frameon = True,
#     alpha = 0.9,
#     show = False,
#     ax = ax
# )   
# ax.spines['top'].set_visible(False)
# ax.spines['right'].set_visible(False)
# plt.title(f'PCA by Model Outcome', pad = 20, fontsize = 14, fontdict={'family':'sans-serif'}, weight = 'bold')
# plt.xlabel('PC1')
# plt.ylabel('PC2')
# plt.savefig(f'{ACC_DATA_ROOT}/bulk_rna_data/outputs/images/PCA_by_Model_outcome.svg',
#             dpi = 600,
#             format = 'svg',
#             bbox_inches = 'tight')
# plt.close()
# # plt.show()

